In [ ]:
import numpy as np
from numpy import ndarray as arr
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path

In [ ]:
# pd_data: list[pd.DataFrame] = [pd.read_csv(f) for f in Path("stocks").glob("**/*")]
pd_data: pd.DataFrame = pd.read_csv("stocks/tsla.csv")

In [ ]:
data: arr = np.array(pd_data)

del pd_data

print(data)

In [ ]:
np.random.shuffle(data)

data[:, 0] = data[:, 0].astype(np.datetime64).astype(np.int64)
data[:, 0] -= np.min(data[:, 0])

# plt.plot(data[:, 0], data[:, 4])

data_test = data[:300]
x_test = data_test[:, [0]].astype(np.int64)
y_test = data_test[:, [4]].astype(np.float64)

data_train = data[300:]
x_train = data_train[:, [0]].astype(np.int64)
y_train = data_train[:, [4]].astype(np.float64)

del data, data_test, data_train

print(x_test.dtype, x_train.dtype, y_test.dtype, y_train.dtype)

In [ ]:
def init_parms() -> list[tuple[arr]]:
    w1: arr = 0.01 * (np.random.rand(1, 5) - 0.5)
    b1: arr = 0.01 * (np.random.rand(1, 5) - 0.5)

    w2: arr = 0.01 * (np.random.rand(5, 50) - 0.5)
    b2: arr = 0.01 * (np.random.rand(1, 50) - 0.5)

    w3: arr = 0.01 * (np.random.rand(50, 100) - 0.5)
    b3: arr = 0.01 * (np.random.rand(1, 100) - 0.5)

    w4: arr = 0.01 * (np.random.rand(100, 25) - 0.5)
    b4: arr = 0.01 * (np.random.rand(1, 25) - 0.5)

    w5: arr = 0.01 * (np.random.rand(25, 1) - 0.5)
    b5: arr = 0.01 * (np.random.rand(1, 1) - 0.5)

    return [(w1, b1), (w2, b2), (w3, b3), (w4, b4), (w5, b5)]


def activationFunction(x: arr, alpha: float = 0.01) -> arr:
    return np.where(x > 0, x, alpha * x)


def activationFunctionDeriv(x: arr, alpha: float = 0.01) -> arr:
    return np.where(x > 0, 1, alpha)


def forward_prop(nn: list[tuple[arr]], x: arr) -> list[tuple[arr]]:
    outputs: list[tuple[arr]] = []

    for i, (w, b) in enumerate(nn):
        inp: arr = outputs[-1][1] if outputs else x


        z: arr = inp @ w + b

        a: arr = activationFunction(z) if i != len(nn) - 1 else z

        outputs.append((z, a))

    return outputs


def back_prop(nn: list[tuple[arr]], outputs: list[arr], x: arr, y: arr) -> list[tuple[arr]]:
    gradient: list[tuple[arr]] = []
    z_deltas: list[arr] = []

    for i in range(len(nn) - 1, -1, -1):
        z, a = outputs[i]

        if z_deltas:
            dz = z_deltas[-1].dot(nn[i + 1][0].T) * activationFunctionDeriv(z)
        else:
            dz = a - y
        
        z_deltas.append(dz)
        
        if i > 0:
            dw = 1 / y.size * dz.T.dot(outputs[i - 1][0])
        else:
            dw = 1 / y.size * dz.T.dot(x)

        db = 1 / y.size * np.sum(dz)

        gradient.append((dw, db))

    return gradient[::-1]

def update_params(nn: list[tuple[arr]], gradient: list[tuple[arr]], learning_rate: float = 0.00001):
    for i, (dw, db) in enumerate(gradient): 
        nn[i] =  (nn[i][0] - learning_rate * dw.T, nn[i][1] - learning_rate * db.T)

def get_error(out: arr, y: arr) -> float:
    return np.mean(np.square(out - y))


def gradient_descend(nn: list[tuple[arr]], x: arr, y: arr, iterations: int = 1000) -> None:
    for i in range(iterations):
        out = forward_prop(nn, x)
        gradient = back_prop(nn, out, x, y)
        # print(y_train[:2], thing[-1][1][:2], '\n')
        update_params(nn, gradient)

        if i % 100 == 0:
            print(f"Iteration {i}: {get_error(out[-1][1], y_train)}")


nn = init_parms()

gradient_descend(nn, x_train, y_train, 10_000)

    

In [ ]:
i = np.argsort(x_test, axis=0).reshape((x_test.size, ))

x = x_test[i]

y = forward_prop(nn, x)[-1][1]

plt.plot(x, y)

other_y = forward_prop(nn, x_train)[-1][1]

j = np.argsort(x_train, axis=0).reshape((x_train.size, ))

plt.plot(x_train[j], y_train[j])

In [ ]:
nn = 10 * init_parms()

_x_test = np.sort(x_test, axis=0)


out = forward_prop(nn, _x_test)[-1][1]

# print(out)

i = np.argsort(x_test, axis=0).reshape((300, ))


#plt.plot(_x_test, y_test[i], label="actual")
plt.plot(_x_test, out, label="predicted")


plt.xlabel("Date")
plt.ylabel("USD")
plt.legend()
plt.show()
